# 12_agarose: Agarose gel analysis

This tutorial is one authoritative HURDLER V2 entry point. It uses a thin notebook and a tested Python backend, records input hashes and limitations, and exports a credential-free workspace.

[Open the current `main` version in Colab](https://colab.research.google.com/github/Wenzhao-protein/clone_repeat_protein/blob/main/notebooks/v2/12_agarose.ipynb)

## 1. Choose a mode and data policy

`tutorial` uses committed fixtures; `colab_full` uses frozen snapshots or explicit refresh/upload; `production_bundle` writes cluster files but never submits; `analyze` consumes finalized compact results. Do not place IDT, Google, GitHub or cluster secrets in `REQUEST`.

In [ ]:
import datetime, importlib.util, os, pathlib, subprocess, sys
if importlib.util.find_spec('hurdler') is None:
    checkout = pathlib.Path('/content/clone_repeat_protein')
    if not checkout.exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', 'main', 'https://github.com/Wenzhao-protein/clone_repeat_protein.git', str(checkout)])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(checkout) + '[notebooks]'])
    os.chdir(checkout)
from IPython.display import JSON, Markdown, display
from hurdler.notebook_workspace import NotebookContext, export_workspace
from hurdler.notebook_backends import agarose as backend
display(Markdown('**HURDLER backend ready.**'))

## 2. Inputs

Edit only this parameter cell for a normal run. Paths may be uploaded Colab files or artifacts imported from a previous workspace.

In [ ]:
MODE = 'tutorial'  # tutorial | colab_full | production_bundle | analyze
SOURCE_MODE = 'snapshot'  # snapshot | refresh | upload
RUN_ID = '12_agarose_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
WORKSPACE_ROOT = '/content/hurdler_workspace' if pathlib.Path('/content').exists() else '/tmp/hurdler_workspace'
REQUEST = {'lane_count': 8}
RANDOM_SEED = 42

In [ ]:
context = NotebookContext(
    run_id=RUN_ID, mode=MODE, source_mode=SOURCE_MODE,
    workspace_root=WORKSPACE_ROOT, random_seed=RANDOM_SEED,
).prepare()
display(JSON(backend.get_spec(), expanded=False))

## 3. Preflight

This stops on a missing input, checksum/schema mismatch or unavailable required external tool. It never silently substitutes another artifact.

In [ ]:
preflight_result = backend.preflight(context, REQUEST)
display(JSON(preflight_result, expanded=False))

## 4. Run

Tutorial work runs in Colab. Heavy production work is exported through notebook 07 and executed on Digs.

In [ ]:
progress_events = []
def on_progress(event):
    progress_events.append(event.to_dict() if hasattr(event, 'to_dict') else dict(event))
result = backend.run(context, REQUEST, progress_callback=on_progress)
backend.write_outputs(context, result)
display(JSON(result.to_dict(), expanded=False))

## 5. Production export

If this workflow declares a production requirement, open notebook 07, select its workflow ID, preview every task, and download the bundle. Colab does not submit jobs or SSH to Digs.

In [ ]:
display(JSON({'progress_event_count': len(progress_events), 'last_event': progress_events[-1] if progress_events else None}, expanded=False))

## 6. Results and provenance

All tables and figures are generated by the backend. The manifest records the repository commit, mode, source policy, warnings, limitations and next notebook IDs.

In [ ]:
workspace_zip = export_workspace(context)
display(Markdown(f'Workspace ready: `{workspace_zip}`'))
try:
    from google.colab import files
except ImportError:
    files = None
# In Colab, uncomment only when you are ready to download:
# if files is not None: files.download(str(workspace_zip))

## 7. Continue

Use `next_notebook_ids` in the result manifest. Keep the exported workspace when moving between notebooks or runtimes.